In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# Local MCP Server

In [18]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command":"python",
            "args": ["2.1_mcp_server.py"]            
        }
    }
)

In [19]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources('local_server')

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [20]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-sonnet-4-5",
    tools=tools,
    system_prompt=prompt
)



In [21]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {
        "messages":[HumanMessage(content="Tell me about OpenClaude")]
    },
    config=config
)

In [22]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about OpenClaude', additional_kwargs={}, response_metadata={}, id='2897ca8b-6db0-47f7-8ecb-263bb21acd33'),
              AIMessage(content=[{'id': 'toolu_014wsuLqzsiDB1StLUcb27mJ', 'caller': {'type': 'direct'}, 'input': {'query': 'openclaude'}, 'name': 'search_web', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01HKiYiNGBWjcWQjWdSQYqzR', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 701, 'output_tokens': 55, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019de717-a0ca-7200-bdcb-0b398f316648-0', tool_calls=[{'name': 'search_web', 'args'

# Online MCP

In [25]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [26]:
agent = create_agent(
    model="claude-sonnet-4-5",
    tools=tools
)

In [27]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='bc5289f4-34bc-48f0-96c1-c255b913ab42'),
              AIMessage(content=[{'text': "I'll get the current time for you in the default timezone (America/New_York).", 'type': 'text'}, {'id': 'toolu_012ay3FbT4feyRB2JGSjKK3g', 'caller': {'type': 'direct'}, 'input': {'timezone': 'America/New_York'}, 'name': 'get_current_time', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01TwgMqV6cRo8KzhBUDQf8CM', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 825, 'output_tokens': 79, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provi

# Travel Agent

In [35]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "kiwi-com-flight-search": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
            
        }
    }
)

tools = await client.get_tools()

In [39]:
tools

[StructuredTool(name='search-flight', description='\n# Search for a flight\n\n## Description\n\nUses the Kiwi API to search for available flights between two locations on a specific date.\n\n## How it works\n\nThe tool will:\n1. Search for matching locations to resolve airport codes\n2. Find available flights for the specified route and date range\n\n## Method\n\nCall this tool whenever a user wants to search for flights, regardless of whether they provided exact airport codes or just city names.\n\nYou should display the returned results in a markdown table format: Group the results by price (those who are the cheapest), duration (those who are the shortest, i.e. have the smallest \'totalDurationInSeconds\') and the rest (those that could still be interesting).\n\nAlways display for each flight in order:\n  - In the 1st column: The departure and arrival airports, including layovers (e.g. "Paris CDG → Barcelona BCN → Lisbon LIS")\n  - In the 2nd column: The departure and arrival dates 

In [47]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="claude-sonnet-4-5",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [48]:

config = {"configurable": {"thread_id": "1"}}

In [49]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Get me a flight from San Francisco to Tokyo on May 22 2026")

response = await agent.ainvoke(
    {"messages": [question]},
    config=config
)

In [50]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a flight from San Francisco to Tokyo on May 22 2026', additional_kwargs={}, response_metadata={}, id='60430104-368e-4038-bee4-bffabb3eb299'),
              AIMessage(content=[{'id': 'toolu_01NbSw1UrSiDbXU5yQNmCf8u', 'caller': {'type': 'direct'}, 'input': {'flyFrom': 'San Francisco', 'flyTo': 'Tokyo', 'departureDate': '22/05/2026'}, 'name': 'search-flight', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01LTBDDYmZaSKSmicFhsusY9', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 2062, 'output_tokens': 97, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id

In [51]:
print(response["messages"][-1].content)

## Flights from San Francisco to Tokyo on May 22, 2026

### ✈️ Cheapest Flights

| Route | Departure → Arrival (Duration) | Class | Price | Book |
|-------|-------------------------------|-------|-------|------|
| SFO → YVR → ICN → NRT | 22/05 18:45 → 25/05 12:55 (50h 10m) | Economy | €632 | [Book](https://on.kiwi.com/20AmsK) |
| SFO → YVR → ICN → NRT | 22/05 18:45 → 25/05 11:05 (48h 20m) | Economy | €635 | [Book](https://on.kiwi.com/bkcgjm) |
| SFO → YVR → ICN → NRT | 22/05 18:45 → 25/05 09:50 (47h 5m) | Economy | €653 | [Book](https://on.kiwi.com/RZP4zF) |

### 🚀 Shortest Flights

| Route | Departure → Arrival (Duration) | Class | Price | Book |
|-------|-------------------------------|-------|-------|------|
| SFO → NRT | 22/05 12:25 → 23/05 15:00 (10h 35m) | Economy | €1,190 | [Book](https://on.kiwi.com/6oZNIb) |
| SFO → NRT | 22/05 12:25 → 23/05 15:00 (10h 35m) | Economy | €1,283 | [Book](https://on.kiwi.com/giTup1) |
| SFO → HND | 22/05 10:55 → 23/05 13:55 (11h) | Economy | €2,33